#### Configuration
Compare greedy TinyStories continuations with and without a midpoint concept push.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / 'llm' / 'model.py').is_file())
sys.path.insert(0, str(REPO_ROOT))

RUN = False
MODEL_NAME = 'roneneldan/TinyStories-28M'
CONCEPT, PROMPT = 'football', 'Once upon a time'
SAMPLE_SIZE, BATCH_SIZE, SEED = 200, 32, 0
MAX_NEW_TOKENS, STEERING_STRENGTH = 96, 6.0
DTYPE, DEVICE_MAP = 'auto', 'auto'
ARTIFACTS = REPO_ROOT / 'artifacts' / MODEL_NAME.rsplit('/', 1)[-1].lower() / '01_concept_direction_steering'
E_PATH = ARTIFACTS / f'E_{CONCEPT}_n{SAMPLE_SIZE}_seed{SEED}.pt'
RESULTS_PATH = ARTIFACTS / f'demo_{CONCEPT}.csv'

#### Model and direction
Reuse the cached direction; row zero is the embedding and block i uses row i + 1.

In [2]:
if RUN:
    import random
    import torch
    from llm.concepts import concept_direction
    from llm.model import blocks, load_model

    random.seed(SEED)
    torch.manual_seed(SEED)
    tokenizer, model = load_model(MODEL_NAME, dtype=DTYPE, device_map=DEVICE_MAP)
    ARTIFACTS.mkdir(parents=True, exist_ok=True)
    if E_PATH.exists():
        direction = torch.load(E_PATH, weights_only=True)
    else:
        direction = concept_direction(model, tokenizer, CONCEPT, SAMPLE_SIZE, BATCH_SIZE)
        torch.save(direction, E_PATH)
    decoder_layers = blocks(model)
    layer_index = len(decoder_layers) // 2
    layer_direction = direction[layer_index + 1]

#### Direct steering
The hook adds the direction to every token at one block during prefill and generation.

In [3]:
def generate_text(strength):
    def add_direction(module, inputs, output):
        hidden = output[0] if isinstance(output, (tuple, list)) else output
        steered = hidden + strength * layer_direction.to(hidden)
        if isinstance(output, tuple):
            return (steered, *output[1:])
        if isinstance(output, list):
            return [steered, *output[1:]]
        return steered

    handle = decoder_layers[layer_index].register_forward_hook(add_direction) if strength else None
    try:
        inputs = tokenizer([PROMPT], return_tensors='pt').to(model.device)
        with torch.inference_mode():
            ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
        continuation = ids[0, inputs.input_ids.shape[-1]:]
        return tokenizer.decode(continuation, skip_special_tokens=True).strip(), len(continuation)
    finally:
        if handle is not None:
            handle.remove()

#### Demonstration
Save both continuations with their steering settings, or read the saved CSV.

In [4]:
if RUN:
    rows = []
    for label, strength in [('Baseline', 0.0), ('Steered', STEERING_STRENGTH)]:
        text, tokens = generate_text(strength)
        rows.append(dict(condition=label, strength=strength, block=layer_index,
                         direction_norm=layer_direction.norm().item(),
                         generated_tokens=tokens, completion=text))
    results = pd.DataFrame(rows)
    results.to_csv(RESULTS_PATH, index=False)
else:
    results = pd.read_csv(RESULTS_PATH)

#### Summary
Read the paired texts alongside the actual direction norm, block, and token counts.

In [5]:
with pd.option_context('display.max_colwidth', None):
    display(results.round(3))

,condition,strength,block,direction_norm,generated_tokens,completion
0,Baseline,0.0,4,0.772,96,", there was a little girl named Lily. She loved to play outside in the sunshine. One day, she saw a big, scary dog. The dog barked and growled at her. Lily was scared and ran away.\n\nLater that day, Lily's mom made her a sandwich. Lily took a bite and made a face. ""This sandwich is disgusting!"" she said. Her mom laughed and said, ""That's okay, Lily. You can have another one"
1,Steered,6.0,4,0.772,96,", there was a little girl named Lily. She loved to play with her toys all day long. One day, she was playing with her toy car when she noticed it was very dirty. She wanted to clean it, so she got a cloth and some water. She wiped the car until it was shiny and new.\n\nLily loved her toy car so much that she wanted to show it to her friends. She went to her friends' house and knocked on the door"


#### Result
A strength-6 push at block 4 changed the 96-token continuation; the concept-vector norm was 0.772. Neither displayed completion is about football, so this direct-push demonstration does not establish concept steering.